# DreamSnap LoRA training worker (Colab GPU)

This long-running worker polls the private training queue, downloads a user's training photos, trains a FLUX.1-dev LoRA with ai-toolkit, uploads the adapter to private S3, and marks the model complete.

Use a separate GPU session from the image-generation worker. Before running, enable a GPU runtime, accept the FLUX.1-dev license on Hugging Face, and add Colab Secrets named `BACKEND_URL`, `WORKER_SECRET`, and `HF_TOKEN`. The backend must use the matching worker secret. Never put secrets in a notebook cell. A free T4 may take hours per training job and can still run out of memory.

In [ ]:
!git clone --depth 1 https://github.com/ostris/ai-toolkit.git /content/ai-toolkit
%cd /content/ai-toolkit
!pip -q install -r requirements.txt

In [ ]:
import json
import os
import shutil
import subprocess
import time
from pathlib import Path

import requests
import torch
import yaml
from google.colab import userdata
from huggingface_hub import login

BACKEND_URL = userdata.get("BACKEND_URL").rstrip("/")
WORKER_SECRET = userdata.get("WORKER_SECRET")
HF_TOKEN = userdata.get("HF_TOKEN")
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime before starting the training worker.")
if not BACKEND_URL.startswith(("https://", "http://localhost", "http://127.0.0.1")):
    raise ValueError("BACKEND_URL must use HTTPS except for a local development server.")

login(token=HF_TOKEN, add_to_git_credential=False)
HEADERS = {"x-worker-secret": WORKER_SECRET}
print(f"Training worker ready on {torch.cuda.get_device_name(0)}.")

In [ ]:
def train_model(job):
    model = job["model"]
    root = Path("/content/dreamsnap-training")
    if root.exists():
        shutil.rmtree(root)
    dataset = root / "dataset"
    dataset.mkdir(parents=True)

    for index, image_url in enumerate(model["trainingImages"]):
        image_response = requests.get(image_url, timeout=120)
        image_response.raise_for_status()
        image_path = dataset / f"subject_{index:03d}.jpg"
        image_path.write_bytes(image_response.content)
        image_path.with_suffix(".txt").write_text("photo of ds_person", encoding="utf-8")

    run_name = f"dreamsnap-{model['id']}"
    config = {
        "job": {
            "name": run_name,
            "process": [{
                "type": "sd_trainer",
                "training_folder": "/content/ai-toolkit/output",
                "device": "cuda:0",
                "network": {"type": "lora", "linear": 16, "linear_alpha": 16},
                "save": {"dtype": "float16", "save_every": 200, "max_step_saves_to_keep": 2},
                "datasets": [{
                    "folder_path": str(dataset),
                    "caption_ext": "txt",
                    "caption_dropout_rate": 0.05,
                    "resolution": [512],
                }],
                "train": {
                    "batch_size": 1,
                    "steps": 600,
                    "gradient_accumulation": 1,
                    "train_unet": True,
                    "train_text_encoder": False,
                    "gradient_checkpointing": True,
                    "noise_scheduler": "flowmatch",
                    "optimizer": "adamw8bit",
                    "lr": 0.0001,
                    "dtype": "fp16",
                },
                "model": {
                    "name_or_path": "black-forest-labs/FLUX.1-dev",
                    "is_flux": True,
                    "quantize": True,
                    "low_vram": True,
                },
                "sample": {
                    "sampler": "flowmatch",
                    "sample_every": 200,
                    "width": 512,
                    "height": 512,
                    "prompts": ["portrait photo of ds_person, natural light"],
                },
            }],
        },
    }
    config_path = root / "config.yaml"
    config_path.parent.mkdir(parents=True, exist_ok=True)
    config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
    environment = os.environ.copy()
    environment["HF_TOKEN"] = HF_TOKEN
    completed = subprocess.run(
        ["python", "run.py", str(config_path)],
        cwd="/content/ai-toolkit",
        env=environment,
        check=False,
    )
    if completed.returncode != 0:
        raise RuntimeError(f"ai-toolkit exited with code {completed.returncode}; inspect the training log above")

    candidates = list(Path("/content/ai-toolkit/output").rglob("*.safetensors"))
    if not candidates:
        raise RuntimeError("Training finished but ai-toolkit produced no .safetensors LoRA file")
    adapter_path = max(candidates, key=lambda path: path.stat().st_mtime)

    ticket = requests.post(
        f"{BACKEND_URL}/worker/training/{job['id']}/upload-url",
        headers=HEADERS,
        timeout=30,
    )
    ticket.raise_for_status()
    upload = ticket.json()
    with adapter_path.open("rb") as adapter_file:
        uploaded = requests.put(
            upload["uploadUrl"],
            data=adapter_file,
            headers={"Content-Type": "application/octet-stream"},
            timeout=600,
        )
    uploaded.raise_for_status()

    result = requests.post(
        f"{BACKEND_URL}/worker/training/{job['id']}/complete",
        headers=HEADERS,
        json={"loraKey": upload["key"]},
        timeout=30,
    )
    result.raise_for_status()
    print(f"Training complete for {model['name']}.")

In [ ]:
def process_training_job(job):
    try:
        train_model(job)
    except Exception as error:
        print(f"Training job {job.get('id')} failed: {error}")
        requests.post(
            f"{BACKEND_URL}/worker/training/{job['id']}/fail",
            headers=HEADERS,
            json={"error": str(error)[:500]},
            timeout=30,
        ).raise_for_status()

def run_training_worker():
    print("Training worker is polling. Interrupt this cell to stop it.")
    while True:
        try:
            response = requests.get(f"{BACKEND_URL}/worker/training-jobs", headers=HEADERS, timeout=30)
            response.raise_for_status()
            job = response.json().get("job")
            if job:
                process_training_job(job)
            else:
                time.sleep(15)
        except KeyboardInterrupt:
            print("Training worker stopped.")
            break
        except Exception as error:
            print(f"Worker polling error: {error}")
            time.sleep(20)

run_training_worker()